# 2 — Propeller Mass Estimation

**Author:** Héctor Fernández Pinacho  
**Project:** IDEAL Lab bachelor thesis pipeline

This notebook estimates the mass of every generated propeller configuration.

The scope is deliberately limited to mass estimation:

1. read the geometry dataset,
2. compute the mass of the hub, blades, and outer ring,
3. export a compact CSV with one row per configuration.

The output file is:

`./csv/prop_mass_estimation.csv`

The exported columns are:

`config_id, m_hub_kg, m_blades_kg, m_outer_ring_kg, m_total_kg`

The `config_id` column is preserved from the geometry dataset. This guarantees that mass row `config_id = n` corresponds to geometry row `config_id = n`.


## 1. Imports

Only numerical, interpolation, and data-handling packages are required.  
No plotting, optimization, or aerodynamic analysis is performed here.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline


## 2. Constants

The notebook is controlled from the constants below.

The constants are separated into two groups:

### Pipeline interface constants

These define how this notebook connects to the geometry-generation stage: file paths, required input columns, and output columns.

### Internal mass-model constants

These are not user-facing sliders. They define the physical and numerical assumptions used to transform geometry into mass.


In [2]:
# =============================================================================
# 1) PIPELINE INTERFACE CONSTANTS
# =============================================================================

CSV_DIR = Path("./csv")

INPUT_GEOMETRY_CSV = CSV_DIR / "prop_geometrical_params.csv"
OUTPUT_MASS_CSV = CSV_DIR / "prop_mass_estimation.csv"


GEOMETRY_COLUMNS = [
    "config_id",
    "radius",
    "ring height",
    "ring thickness",
    "blade count",
    "inner thickness",
    "inner max pos",
    "inner camber",
    "inner chord",
    "inner angle",
    "mid radial pos",
    "mid chord",
    "mid angle",
    "outer thickness",
    "outer max pos",
    "outer camber",
    "outer chord",
    "outer angle",
]


MASS_OUTPUT_COLUMNS = [
    "config_id",
    "m_hub_kg",
    "m_blades_kg",
    "m_outer_ring_kg",
    "m_total_kg",
]


# Convenience column aliases.
CONFIG_ID_COL = "config_id"

RADIUS_COL = "radius"
RING_HEIGHT_COL = "ring height"
RING_THICKNESS_COL = "ring thickness"
BLADE_COUNT_COL = "blade count"

INNER_THICKNESS_COL = "inner thickness"
INNER_CHORD_COL = "inner chord"

MID_RADIAL_POS_COL = "mid radial pos"
MID_CHORD_COL = "mid chord"

OUTER_THICKNESS_COL = "outer thickness"
OUTER_CHORD_COL = "outer chord"


# =============================================================================
# 2) INTERNAL MASS-MODEL CONSTANTS
# =============================================================================

# -----------------------------
# Radial stations
# -----------------------------
# The inner profile is a loft/control profile located inside the hub.
# The visible blade mass starts at the hub outer radius.
INNER_STATION_RADIUS_MM = 4.5
HUB_OUTER_RADIUS_MM = 8.25


# -----------------------------
# Printed-material calibration
# -----------------------------
# A 20 x 20 x 20 mm calibration cube printed with the same slicer settings is
# used to estimate the effective printed density.
CALIBRATION_CUBE_SIDE_MM = 20.0
CALIBRATION_CUBE_MASS_G = 6.2


# The hub mass is treated as a measured constant because the hub geometry does
# not change across the generated propeller configurations.
HUB_MASS_MEASUREMENTS_G = [
    1.9 / 3,
    1.9 / 3,
    1.9 / 3,
]


# -----------------------------
# Blade section model
# -----------------------------
# NACA 4-digit symmetric area approximation:
#
#     A_section = K * chord^2 * thickness_fraction
#
# The thickness sliders are expressed as percentage of chord.
NACA_SECTION_AREA_COEFFICIENT = 0.6851


# Chord and thickness are represented with natural cubic splines through the
# inner, middle, and outer profile stations.
BLADE_SPLINE_BC_TYPE = "natural"


# Deterministic Gauss-Legendre integration points for the blade volume.
BLADE_INTEGRATION_POINTS = 64


# -----------------------------
# Outer ring model
# -----------------------------
# The outer ring is modelled as an elliptical cross-section swept along the
# circumference at r = radius.
RING_CROSS_SECTION_SHAPE = "ellipse"


# -----------------------------
# Output formatting
# -----------------------------
ROUND_MASS_KG_DECIMALS = 8


## 3. Load and validate the geometry input

The geometry CSV must contain one row per propeller configuration and all geometry columns required by the mass model.

The most important identifier is `config_id`. It is copied directly into the mass output so that the mass dataset can always be joined back to the geometry dataset without relying only on row order.


In [3]:
def load_geometry_dataframe(path):
    """Load the geometry CSV and check that all required columns are present."""
    if not path.exists():
        raise FileNotFoundError(
            f"Input geometry file not found: {path}. "
            "Run the geometry-generation notebook first."
        )

    df = pd.read_csv(path)

    missing_columns = [column for column in GEOMETRY_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(
            "The geometry CSV is missing the following required columns: "
            + ", ".join(missing_columns)
        )

    df = df[GEOMETRY_COLUMNS].copy()

    if df[CONFIG_ID_COL].duplicated().any():
        raise ValueError("Duplicate config_id values were found in the geometry CSV.")

    return df


geometry_df = load_geometry_dataframe(INPUT_GEOMETRY_CSV)

print(f"Loaded {len(geometry_df)} propeller configurations from:")
print(f"  {INPUT_GEOMETRY_CSV}")
print()
print("First config_id values:")
print(geometry_df[CONFIG_ID_COL].head().to_list())


Loaded 5000 propeller configurations from:
  csv\prop_geometrical_params.csv

First config_id values:
[0, 1, 2, 3, 4]


## 4. Density and constant hub mass

The effective printed density is estimated from a calibration cube printed with the same slicer settings as the propellers.

For a cube with side length \(L\), the volume is:

$$
V_{cube} = L^3
$$

and the effective density is:

$$
\rho_{eff} = \frac{m_{cube}}{V_{cube}}
$$

The hub is treated as a constant measured component. Since the hub does not vary between configurations, the same hub mass is assigned to every row.


In [4]:
def cube_density_g_per_cm3(cube_mass_g, cube_side_mm):
    """Return effective printed density in g/cm^3."""
    cube_side_cm = cube_side_mm / 10.0
    cube_volume_cm3 = cube_side_cm ** 3
    return cube_mass_g / cube_volume_cm3


RHO_EFFECTIVE_G_PER_CM3 = cube_density_g_per_cm3(
    cube_mass_g=CALIBRATION_CUBE_MASS_G,
    cube_side_mm=CALIBRATION_CUBE_SIDE_MM,
)

RHO_EFFECTIVE_KG_PER_MM3 = RHO_EFFECTIVE_G_PER_CM3 * 1e-6

HUB_MASS_KG = float(np.mean(HUB_MASS_MEASUREMENTS_G)) / 1000.0
HUB_MASS_STD_KG = float(np.std(HUB_MASS_MEASUREMENTS_G)) / 1000.0

print(f"Effective printed density : {RHO_EFFECTIVE_G_PER_CM3:.4f} g/cm^3")
print(f"Hub mass                  : {HUB_MASS_KG:.8f} kg")
print(f"Hub mass std              : {HUB_MASS_STD_KG:.8f} kg")


Effective printed density : 0.7750 g/cm^3
Hub mass                  : 0.00063333 kg
Hub mass std              : 0.00000000 kg


## 5. Blade mass model

Each blade is represented by three radial profile stations:

1. inner profile,
2. middle profile,
3. outer profile.

The inner profile is located inside the hub and acts as a loft-control profile. The CAD blade is generated by lofting smoothly between profiles, so the mass model also uses a smooth radial interpolation.

For mass estimation, the blade volume inside the hub must not be counted. Therefore, the spline is fitted from the inner station to the outer station, but the volume integral starts only at the hub outer radius:

$$
V_{blade}
=
\int_{R_{hub}}^{R}
A(r)\,dr
$$

where \(R_{hub} = 8.25\,\mathrm{mm}\) and \(R\) is the propeller radius.

The cross-sectional area is approximated as:

$$
A(r)
=
K_{NACA} \, c(r)^2 \, t(r)
$$

where:

- \(c(r)\) is the chord length in mm,
- \(t(r)\) is the thickness as a fraction of chord,
- \(K_{NACA}\) is the airfoil section area coefficient.

The dataset contains inner and outer thickness values. The middle thickness is inferred at the actual middle radial station and then used together with the inner and outer thickness values to build the cubic spline.


In [5]:
def section_area_mm2(chord_mm, thickness_fraction):
    """Approximate NACA-like airfoil section area in mm^2."""
    return (
        NACA_SECTION_AREA_COEFFICIENT
        * chord_mm ** 2
        * thickness_fraction
    )


def compute_middle_thickness_percent(
    inner_thickness_percent,
    outer_thickness_percent,
    inner_radius_mm,
    middle_radius_mm,
    outer_radius_mm,
):
    """Infer the middle thickness percentage at the middle radial station."""
    radial_fraction = (
        (middle_radius_mm - inner_radius_mm)
        / (outer_radius_mm - inner_radius_mm)
    )

    return (
        inner_thickness_percent
        + radial_fraction * (outer_thickness_percent - inner_thickness_percent)
    )


def blade_volume_one_blade_mm3(row):
    """Estimate the visible volume of one blade in mm^3."""
    outer_radius_mm = float(row[RADIUS_COL])
    middle_radius_mm = float(row[MID_RADIAL_POS_COL]) * outer_radius_mm

    valid_ordering = (
        INNER_STATION_RADIUS_MM
        < HUB_OUTER_RADIUS_MM
        < middle_radius_mm
        < outer_radius_mm
    )

    if not valid_ordering:
        raise ValueError(
            "Invalid radial station ordering. Expected: "
            "inner station < hub outer radius < middle station < outer radius."
        )

    inner_thickness_percent = float(row[INNER_THICKNESS_COL])
    outer_thickness_percent = float(row[OUTER_THICKNESS_COL])

    middle_thickness_percent = compute_middle_thickness_percent(
        inner_thickness_percent=inner_thickness_percent,
        outer_thickness_percent=outer_thickness_percent,
        inner_radius_mm=INNER_STATION_RADIUS_MM,
        middle_radius_mm=middle_radius_mm,
        outer_radius_mm=outer_radius_mm,
    )

    radial_stations_mm = np.array([
        INNER_STATION_RADIUS_MM,
        middle_radius_mm,
        outer_radius_mm,
    ])

    chord_values_mm = np.array([
        float(row[INNER_CHORD_COL]),
        float(row[MID_CHORD_COL]),
        float(row[OUTER_CHORD_COL]),
    ])

    thickness_values_fraction = np.array([
        inner_thickness_percent,
        middle_thickness_percent,
        outer_thickness_percent,
    ]) / 100.0

    chord_spline = CubicSpline(
        radial_stations_mm,
        chord_values_mm,
        bc_type=BLADE_SPLINE_BC_TYPE,
    )

    thickness_spline = CubicSpline(
        radial_stations_mm,
        thickness_values_fraction,
        bc_type=BLADE_SPLINE_BC_TYPE,
    )

    lower_limit_mm = HUB_OUTER_RADIUS_MM
    upper_limit_mm = outer_radius_mm

    gauss_nodes, gauss_weights = np.polynomial.legendre.leggauss(
        BLADE_INTEGRATION_POINTS
    )

    evaluation_radii_mm = (
        0.5 * (upper_limit_mm - lower_limit_mm) * gauss_nodes
        + 0.5 * (upper_limit_mm + lower_limit_mm)
    )

    chord_at_r_mm = chord_spline(evaluation_radii_mm)
    thickness_at_r = thickness_spline(evaluation_radii_mm)

    area_at_r_mm2 = section_area_mm2(
        chord_mm=chord_at_r_mm,
        thickness_fraction=thickness_at_r,
    )

    blade_volume_mm3 = (
        0.5
        * (upper_limit_mm - lower_limit_mm)
        * np.sum(gauss_weights * area_at_r_mm2)
    )

    return float(blade_volume_mm3)


def total_blades_volume_mm3(row):
    """Return the total visible blade volume of one propeller in mm^3."""
    return (
        int(row[BLADE_COUNT_COL])
        * blade_volume_one_blade_mm3(row)
    )


## 6. Outer ring mass model

The outer ring is modelled as an elliptical cross-section swept around a circular path at the propeller radius.

The ellipse semi-axes are:

$$
a = \frac{h}{2}
$$

$$
b = \frac{t}{2}
$$

so the cross-sectional area is:

$$
A_{ring} = \pi a b
$$

The swept length is the circumference at \(r = R\):

$$
L = 2 \pi R
$$

Therefore, the ring volume is:

$$
V_{ring}
=
\pi \frac{h}{2} \frac{t}{2} \, 2 \pi R
=
\frac{\pi^2}{2} R h t
$$


In [6]:
def outer_ring_volume_mm3(row):
    """Estimate the outer ring volume in mm^3 using an elliptical cross-section."""
    radius_mm = float(row[RADIUS_COL])
    height_mm = float(row[RING_HEIGHT_COL])
    thickness_mm = float(row[RING_THICKNESS_COL])

    return (
        (np.pi ** 2 / 2.0)
        * radius_mm
        * height_mm
        * thickness_mm
    )


## 7. Compute and export masses

The blade and outer ring masses are obtained from volume and effective density:

$$
m = V \rho_{eff}
$$

The total mass is computed as:

$$
m_{total}
=
m_{hub}
+
m_{blades}
+
m_{outer\,ring}
$$

The component masses are rounded first, and then the total is recomputed from the rounded component values. This ensures that the exported total mass matches the exported component sum.


In [7]:
def compute_mass_dataframe(geometry_df):
    """Compute hub, blade, outer-ring, and total propeller mass in kg."""
    blade_volumes_mm3 = geometry_df.apply(total_blades_volume_mm3, axis=1)
    outer_ring_volumes_mm3 = geometry_df.apply(outer_ring_volume_mm3, axis=1)

    mass_df = pd.DataFrame(index=geometry_df.index)

    mass_df[CONFIG_ID_COL] = geometry_df[CONFIG_ID_COL].astype(int).to_numpy()

    mass_df["m_hub_kg"] = HUB_MASS_KG
    mass_df["m_blades_kg"] = blade_volumes_mm3 * RHO_EFFECTIVE_KG_PER_MM3
    mass_df["m_outer_ring_kg"] = outer_ring_volumes_mm3 * RHO_EFFECTIVE_KG_PER_MM3

    mass_columns = [
        "m_hub_kg",
        "m_blades_kg",
        "m_outer_ring_kg",
    ]

    mass_df[mass_columns] = mass_df[mass_columns].round(ROUND_MASS_KG_DECIMALS)

    mass_df["m_total_kg"] = (
        mass_df["m_hub_kg"]
        + mass_df["m_blades_kg"]
        + mass_df["m_outer_ring_kg"]
    ).round(ROUND_MASS_KG_DECIMALS)

    return mass_df[MASS_OUTPUT_COLUMNS]


mass_df = compute_mass_dataframe(geometry_df)

CSV_DIR.mkdir(parents=True, exist_ok=True)
mass_df.to_csv(OUTPUT_MASS_CSV, index=False)

print(f"Mass estimation completed for {len(mass_df)} configurations.")
print(f"Saved output CSV to:")
print(f"  {OUTPUT_MASS_CSV}")


Mass estimation completed for 5000 configurations.
Saved output CSV to:
  csv\prop_mass_estimation.csv


## 8. Final checks

This section verifies that the exported dataset is ready for downstream use.

The checks confirm that:

- the output has exactly the expected columns,
- `config_id` has been preserved correctly,
- every mass value is finite and positive,
- the total mass equals the sum of the three exported component masses,
- the CSV file exists at the expected location.


In [8]:
def run_final_checks(mass_df, geometry_df, output_csv):
    """Validate the mass output before using it downstream."""
    all_ok = True

    if list(mass_df.columns) == MASS_OUTPUT_COLUMNS:
        print("[ OK ] Output columns match the required structure.")
    else:
        print("[FAIL] Output columns do not match the required structure.")
        all_ok = False

    if mass_df[CONFIG_ID_COL].equals(geometry_df[CONFIG_ID_COL].astype(int)):
        print("[ OK ] config_id matches the geometry input.")
    else:
        print("[FAIL] config_id does not match the geometry input.")
        all_ok = False

    numeric_mass_columns = [
        "m_hub_kg",
        "m_blades_kg",
        "m_outer_ring_kg",
        "m_total_kg",
    ]

    if np.isfinite(mass_df[numeric_mass_columns].to_numpy()).all():
        print("[ OK ] All mass values are finite.")
    else:
        print("[FAIL] Some mass values are not finite.")
        all_ok = False

    if (mass_df[numeric_mass_columns] > 0).all().all():
        print("[ OK ] All mass values are positive.")
    else:
        print("[FAIL] Some mass values are not positive.")
        all_ok = False

    recomputed_total = (
        mass_df["m_hub_kg"]
        + mass_df["m_blades_kg"]
        + mass_df["m_outer_ring_kg"]
    ).round(ROUND_MASS_KG_DECIMALS)

    total_error = (
        mass_df["m_total_kg"]
        - recomputed_total
    ).abs().max()

    tolerance = 10 ** (-ROUND_MASS_KG_DECIMALS)

    if total_error <= tolerance:
        print("[ OK ] Total mass equals hub + blades + outer ring.")
    else:
        print("[FAIL] Total mass does not match component sum.")
        print(f"       Maximum absolute error: {total_error:.12f} kg")
        all_ok = False

    if output_csv.exists():
        print("[ OK ] CSV file was exported successfully.")
    else:
        print("[FAIL] CSV file was not found after export.")
        all_ok = False

    print()
    print("Mass summary [kg]")
    print("-" * 72)
    print(mass_df[numeric_mass_columns].describe().loc[["min", "mean", "max"]])

    print()
    print("=" * 72)
    print("ALL CHECKS PASSED ✓" if all_ok else "SOME CHECKS FAILED")
    print("=" * 72)


run_final_checks(mass_df, geometry_df, OUTPUT_MASS_CSV)


[ OK ] Output columns match the required structure.
[ OK ] config_id matches the geometry input.
[ OK ] All mass values are finite.
[ OK ] All mass values are positive.
[ OK ] Total mass equals hub + blades + outer ring.
[ OK ] CSV file was exported successfully.

Mass summary [kg]
------------------------------------------------------------------------
      m_hub_kg  m_blades_kg  m_outer_ring_kg  m_total_kg
min   0.000633     0.001183         0.000918    0.003226
mean  0.000633     0.007948         0.004493    0.013074
max   0.000633     0.031843         0.015298    0.040531

ALL CHECKS PASSED ✓
